# 🔬 SYSTEMATIC RESEARCH ENGINE
## 353 Tickers → 50 Best Opportunities

**Mission:** Screen all your stocks (holdings + watchlists) using FREE data + GPU analysis

**Your Universe (353 Tickers):**
- 9 Current Holdings: IONQ, ASTS, APLD, HOOD, UBER, LYFT, LUNR, XBIO, KDK
- Alpha 76 Watchlist: SERV, PALI, RGTI, QUBT, SOUN, MARA, RIOT, SMCI, etc.
- 204 Small Caps across 15 sectors
- Coverage: Tech, Healthcare, Aerospace, Energy, Fintech, Crypto, Biotech

**Pipeline:**
1. Load 353 tickers → 2. OHLCV (yfinance) → 3. Fundamentals (Yahoo scraping) → 4. Insider trades (SEC) → 5. News sentiment (Google + GPU) → 6. Score & rank → 7. Top 50

**Data Sources (100% FREE):**
✅ yfinance, Yahoo HTML scraping, SEC EDGAR JSON, Google News RSS, GPU FinBERT

---

In [ ]:
# Install required packages
!pip install -q yfinance pandas numpy requests beautifulsoup4 lxml
!pip install -q torch transformers sentence-transformers
!pip install -q matplotlib seaborn plotly

print("✅ Packages installed")

In [ ]:
# Import all libraries
import pandas as pd
import numpy as np
import yfinance as yf
import requests
from bs4 import BeautifulSoup
import time
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# GPU check
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ Running on CPU")

print("\n✅ All imports successful")

In [ ]:
# Browser headers (crucial for scraping)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# Auto-detect environment and set paths
import os
import sys

# Check if running in Codespace (Linux) or local (Windows/Mac)
if os.path.exists('/workspaces/quantum-ai-trader_v1.1'):
    # Codespace environment
    WORKSPACE_ROOT = '/workspaces/quantum-ai-trader_v1.1'
elif os.path.exists(r'C:\Users\alexj\Desktop\shadow_ai\quantum-ai-trader_v1.1'):
    # Shadow PC (original path)
    WORKSPACE_ROOT = r'C:\Users\alexj\Desktop\shadow_ai\quantum-ai-trader_v1.1'
elif os.path.exists(r'C:\Users\Shadow\quantum-ai-trader_v1.1'):
    # Shadow PC (alternate path)
    WORKSPACE_ROOT = r'C:\Users\Shadow\quantum-ai-trader_v1.1'
else:
    # Try relative path from notebook location
    notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
    WORKSPACE_ROOT = os.path.abspath(os.path.join(notebook_dir, '..'))

DATA_DIR = os.path.join(WORKSPACE_ROOT, 'data')
UNIVERSE_FILE = os.path.join(DATA_DIR, 'ticker_universe_300.csv')
OUTPUT_FILE = os.path.join(DATA_DIR, 'research_results_top50.csv')

# Verify paths exist
print(f"✅ Workspace: {WORKSPACE_ROOT}")
print(f"✅ Data directory: {DATA_DIR}")
print(f"✅ Universe file: {UNIVERSE_FILE}")
print(f"✅ File exists: {os.path.exists(UNIVERSE_FILE)}")

if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)
    print(f"📁 Created data directory")

## 📋 PHASE 2: Load Ticker Universe

In [ ]:
# IF FILE DOESN'T EXIST: Download from GitHub or create sample
if not os.path.exists(UNIVERSE_FILE):
    print("⚠️ Universe file not found. Creating sample...")
    
    # Option 1: Try to download from GitHub
    import requests
    try:
        url = "https://raw.githubusercontent.com/alexpayne556-collab/quantum-ai-trader_v1.1/main/data/ticker_universe_300.csv"
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            with open(UNIVERSE_FILE, 'wb') as f:
                f.write(response.content)
            print(f"✅ Downloaded from GitHub to {UNIVERSE_FILE}")
        else:
            raise Exception("GitHub download failed")
    except:
        # Option 2: Create minimal sample for testing
        print("⚠️ Creating minimal sample (replace with full list later)")
        sample_tickers = [
            ('AAPL', 'Technology', 'Consumer Electronics', 'Large Cap', 'Benchmark'),
            ('NVDA', 'Technology', 'Semiconductors', 'Large Cap', 'AI Leader'),
            ('IONQ', 'Technology', 'Quantum Computing', 'Small Cap', 'Current Holding'),
            ('ASTS', 'Communications', 'Space', 'Small Cap', 'Current Holding'),
            ('HOOD', 'Fintech', 'Brokerage', 'Mid Cap', 'Current Holding'),
            ('MARA', 'Crypto', 'Bitcoin Mining', 'Small Cap', 'Mining Leader'),
            ('PLTR', 'Technology', 'Software', 'Mid Cap', 'AI/Data'),
            ('TSLA', 'Automotive', 'Electric Vehicles', 'Large Cap', 'Innovation'),
            ('AMD', 'Technology', 'Semiconductors', 'Large Cap', 'AI/GPU'),
            ('SOFI', 'Fintech', 'Banking', 'Small Cap', 'Digital Bank'),
        ]
        import pandas as pd
        df = pd.DataFrame(sample_tickers, columns=['ticker', 'sector', 'industry', 'market_cap_category', 'notes'])
        df.to_csv(UNIVERSE_FILE, index=False)
        print(f"✅ Created sample file: {UNIVERSE_FILE}")
        print("⚠️ IMPORTANT: Replace this with your full 353-ticker list!")
else:
    print("✅ Universe file found")

In [ ]:
# Load 300-ticker universe
universe_df = pd.read_csv(UNIVERSE_FILE)

print(f"📊 Loaded {len(universe_df)} tickers\n")
print("Sector breakdown:")
print(universe_df['sector'].value_counts())
print("\nFirst 10 tickers:")
print(universe_df.head(10))

In [ ]:
# Get ticker list
TICKERS = universe_df['ticker'].tolist()

print(f"✅ {len(TICKERS)} tickers ready for analysis")
print(f"\nSample: {TICKERS[:20]}")

## 📈 PHASE 3: OHLCV Data Collection (yfinance)

In [ ]:
def collect_ohlcv_batch(tickers, period='2y', batch_size=50):
    """
    Collect OHLCV data using yfinance (unlimited, free)
    Batch processing to avoid memory issues
    """
    all_data = {}
    failed = []
    
    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i+batch_size]
        print(f"Fetching batch {i//batch_size + 1} ({len(batch)} tickers)...")
        
        try:
            # Download batch (yfinance handles multiple tickers efficiently)
            data = yf.download(batch, period=period, group_by='ticker', 
                             progress=False, threads=True)
            
            # Store each ticker's data
            for ticker in batch:
                try:
                    if len(batch) == 1:
                        ticker_data = data
                    else:
                        ticker_data = data[ticker]
                    
                    if not ticker_data.empty and len(ticker_data) > 20:
                        all_data[ticker] = ticker_data
                    else:
                        failed.append(ticker)
                except:
                    failed.append(ticker)
            
            time.sleep(0.5)  # Polite delay
            
        except Exception as e:
            print(f"  Error in batch: {e}")
            failed.extend(batch)
    
    print(f"\n✅ Collected: {len(all_data)}/{len(tickers)} tickers")
    if failed:
        print(f"❌ Failed: {len(failed)} tickers - {failed[:10]}")
    
    return all_data, failed

# Test with first 20 tickers
print("🔄 Testing OHLCV collection with 20 tickers...")
test_tickers = TICKERS[:20]
ohlcv_data, failed_tickers = collect_ohlcv_batch(test_tickers, period='1y')

# Show sample
if ohlcv_data:
    sample_ticker = list(ohlcv_data.keys())[0]
    print(f"\nSample data for {sample_ticker}:")
    print(ohlcv_data[sample_ticker].tail())

## 🔍 PHASE 4: Yahoo Finance Fundamentals Scraper

In [ ]:
import logging
from typing import Dict, Any, Optional
from datetime import datetime

# ==================== LOAD API KEYS ====================
load_dotenv()
FMP_API_KEY = os.getenv('FMP_API_KEY', 'MISSING')
ALPHA_VANTAGE_API_KEY = os.getenv('ALPHA_VANTAGE_API_KEY', 'MISSING')

# ==================== CONFIGURE LOGGING ====================
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    handlers=[logging.FileHandler('data/fundamentals_collector.log'),
                              logging.StreamHandler()])
logger = logging.getLogger(__name__)

# ==================== PRIMARY SOURCE: yfinance ====================
def fetch_yfinance(ticker: str) -> Dict[str, Any]:
    """Fetch data from yfinance. Returns dict with source attribution."""
    data = {}
    source = "yfinance"
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        # Map yfinance fields to standardized names
        field_mapping = {
            'marketCap': 'market_cap',
            'trailingPE': 'trailing_pe',
            'forwardPE': 'forward_pe',
            'shortPercentOfFloat': 'short_pct',
            'heldPercentInsiders': 'insider_pct',
            'heldPercentInstitutions': 'institution_pct',
            'beta': 'beta',
            'priceToBook': 'price_to_book',
            'priceToSalesTrailing12Months': 'price_to_sales',
            'pegRatio': 'peg_ratio'
        }

        for yf_field, our_field in field_mapping.items():
            value = info.get(yf_field)
            if value is not None and value != 'N/A' and value != -99:
                data[our_field] = {'value': value, 'source': source}

        data['_last_updated'] = datetime.utcnow().isoformat()
        logger.info(f"[{ticker}] Retrieved {len(data)-1} fields from {source}")
        return data

    except Exception as e:
        logger.warning(f"[{ticker}] Failed to get data from {source}: {e}")
        return {}

# ==================== FALLBACK SOURCE 1: FMP API ====================
def fetch_fmp(ticker: str, existing_data: Dict[str, Any]) -> Dict[str, Any]:
    """Fetch missing data from Financial Modeling Prep API."""
    source = "fmp"
    fetched_data = {}
    try:
        logger.info(f"[{ticker}] Trying {source} fallback...")
        url = f"https://financialmodelingprep.com/api/v3/profile/{ticker}?apikey={FMP_API_KEY}"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        company_data = response.json()

        if isinstance(company_data, list) and len(company_data) > 0:
            profile = company_data[0]
            
            # Only fetch missing fields
            fmp_field_mapping = {
                'priceEarningsToGrowthRatio': 'peg_ratio',
                'beta': 'beta',
                'priceToBook': 'price_to_book',
            }
            for fmp_field, our_field in fmp_field_mapping.items():
                if our_field not in existing_data and fmp_field in profile:
                    value = profile.get(fmp_field)
                    if value is not None:
                        fetched_data[our_field] = {'value': value, 'source': source}

        time.sleep(0.2)
        if fetched_data:
            logger.info(f"[{ticker}] Added {len(fetched_data)} fields from {source}")
        return fetched_data

    except requests.exceptions.RequestException as e:
        logger.warning(f"[{ticker}] Network error from {source}: {e}")
        return {}
    except Exception as e:
        logger.warning(f"[{ticker}] Failed to parse data from {source}: {e}")
        return {}

# ==================== FALLBACK SOURCE 2: Alpha Vantage ====================
def fetch_alpha_vantage(ticker: str, existing_data: Dict[str, Any]) -> Dict[str, Any]:
    """Fetch missing data from Alpha Vantage API (sparingly - 25 req/day limit)."""
    source = "alpha_vantage"
    fetched_data = {}
    try:
        # Only call if we're still missing PEG
        if 'peg_ratio' in existing_data:
            return {}

        logger.info(f"[{ticker}] Trying {source} for PEG ratio...")
        url = f"https://www.alphavantage.co/query"
        params = {
            'function': 'OVERVIEW',
            'symbol': ticker,
            'apikey': ALPHA_VANTAGE_API_KEY
        }
        response = requests.get(url, params=params, timeout=15)
        response.raise_for_status()
        overview_data = response.json()

        if "Error Message" in overview_data:
            logger.warning(f"[{ticker}] {source} returned error: {overview_data['Error Message']}")
            return {}

        # Extract PEG ratio
        if 'PEGRatio' in overview_data and overview_data['PEGRatio'] not in ('', 'None', None):
            value = overview_data.get('PEGRatio')
            try:
                fetched_data['peg_ratio'] = {'value': float(value), 'source': source}
                logger.info(f"[{ticker}] Found PEG ratio from {source}")
            except (ValueError, TypeError):
                pass

        time.sleep(15)  # Strict rate limiting (5 calls/min)
        return fetched_data

    except requests.exceptions.RequestException as e:
        logger.warning(f"[{ticker}] Network error from {source}: {e}")
        return {}
    except Exception as e:
        logger.warning(f"[{ticker}] Failed to parse data from {source}: {e}")
        return {}

# ==================== MASTER COLLECTOR FUNCTION ====================
def collect_fundamentals_multisource(ticker: str) -> Dict[str, Any]:
    """
    Multi-source collector with fallbacks: yfinance → FMP → Alpha Vantage
    Returns dict with all fields or N/A with source attribution.
    """
    all_data = {}
    logger.info(f"=== Starting collection for {ticker} ===")

    # 1. PRIMARY: yfinance
    all_data.update(fetch_yfinance(ticker))

    # 2. FALLBACK 1: FMP (for missing fields, especially PEG)
    missing_critical = any(field not in all_data for field in ['peg_ratio', 'trailing_pe'])
    if missing_critical:
        fmp_data = fetch_fmp(ticker, all_data)
        all_data.update(fmp_data)

    # 3. FALLBACK 2: Alpha Vantage (specifically for PEG if still missing)
    if 'peg_ratio' not in all_data:
        av_data = fetch_alpha_vantage(ticker, all_data)
        all_data.update(av_data)

    # 4. Ensure all expected fields exist
    expected_fields = [
        'market_cap', 'trailing_pe', 'forward_pe', 'peg_ratio',
        'short_pct', 'insider_pct', 'institution_pct', 'beta', 'price_to_book', 'price_to_sales'
    ]
    final_output = {'ticker': ticker}
    for field in expected_fields:
        final_output[field] = all_data.get(field, {'value': None, 'source': None})

    logger.info(f"=== Completed {ticker}. Fields found: {sum(1 for f in expected_fields if final_output[f]['value'] is not None)}/{len(expected_fields)} ===\n")
    return final_output

# ==================== TEST ON YOUR HOLDINGS ====================
print("✅ API Keys loaded:")
print(f"  FMP: {'✓' if FMP_API_KEY != 'MISSING' else '❌'}")
print(f"  Alpha Vantage: {'✓' if ALPHA_VANTAGE_API_KEY != 'MISSING' else '❌'}\n")

print("🔄 Testing multi-source fundamentals collector on YOUR holdings...")
test_tickers_fund = ['IONQ', 'ASTS', 'HOOD', 'MARA', 'PLTR']
test_funds = []

for ticker in test_tickers_fund:
    print(f"\n  Processing {ticker}...")
    fund = collect_fundamentals_multisource(ticker)
    test_funds.append(fund)
    time.sleep(1)

# Flatten results for display
flat_results = []
for r in test_funds:
    flat_row = {'ticker': r['ticker']}
    for field in ['market_cap', 'trailing_pe', 'peg_ratio', 'short_pct', 'insider_pct', 'beta']:
        data = r.get(field, {})
        flat_row[f"{field}_value"] = data.get('value')
        flat_row[f"{field}_source"] = data.get('source')
    flat_results.append(flat_row)

fund_df = pd.DataFrame(flat_results)
print("\n📊 Results (with source attribution):")
print(fund_df)

# Summary
print("\n✅ Multi-source fundamentals scraper tested successfully!")
print(f"   Processed: {len(test_funds)} tickers")
print(f"   Fields per ticker: {sum(1 for f in ['market_cap', 'trailing_pe', 'peg_ratio', 'short_pct', 'insider_pct', 'beta'] if any(r.get(f, {}).get('value') is not None for r in test_funds))}/6")

## 🕵️ PHASE 5: SEC EDGAR Insider Trading Scraper

## ⚙️ PHASE 4B: Batch Processor with Checkpoints (353 Tickers)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import json

class FundamentalDataBatchProcessor:
    """Batch processor with checkpointing for 353 tickers. Runs in 45-60 minutes."""
    
    def __init__(self, tickers, max_workers=5, checkpoint_interval=50):
        self.tickers = tickers
        self.max_workers = max_workers
        self.checkpoint_interval = checkpoint_interval
        self.results = []
        self.failed_tickers = []
        self.progress_file = "data/fundamentals_checkpoint.json"
        self.final_output_file = f"data/results/fundamentals_batch_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
        
        # Create directories if needed
        os.makedirs('data/results', exist_ok=True)

    def load_checkpoint(self):
        """Load previous progress from checkpoint file."""
        try:
            with open(self.progress_file, 'r') as f:
                saved = json.load(f)
                self.results = saved.get('results', [])
                processed_set = {r['ticker'] for r in self.results}
                # Only process new tickers
                self.tickers = [t for t in self.tickers if t not in processed_set]
                logger.info(f"Loaded checkpoint. {len(self.results)} already processed. {len(self.tickers)} remaining.")
        except FileNotFoundError:
            logger.info("No checkpoint found. Starting fresh.")
        except Exception as e:
            logger.error(f"Error loading checkpoint: {e}. Starting fresh.")

    def save_checkpoint(self):
        """Save current progress to checkpoint file."""
        try:
            checkpoint_data = {
                'timestamp': datetime.utcnow().isoformat(),
                'results': self.results,
                'failed': self.failed_tickers
            }
            with open(self.progress_file, 'w') as f:
                json.dump(checkpoint_data, f, indent=2)
            logger.info(f"Checkpoint saved. {len(self.results)} processed, {len(self.failed_tickers)} failed.")
        except Exception as e:
            logger.error(f"Failed to save checkpoint: {e}")

    def process_single(self, ticker: str):
        """Wrapper with retry logic (exponential backoff)."""
        max_retries = 3
        for attempt in range(max_retries):
            try:
                result = collect_fundamentals_multisource(ticker)
                return result
            except Exception as e:
                if attempt == max_retries - 1:
                    raise e
                wait_time = (2 ** attempt) + 1
                logger.warning(f"[{ticker}] Attempt {attempt+1} failed. Retrying in {wait_time}s.")
                time.sleep(wait_time)

    def run_batch(self):
        """Main batch processor with parallel execution and progress tracking."""
        logger.info(f"Starting batch processing for {len(self.tickers)} tickers with {self.max_workers} workers.")
        self.load_checkpoint()

        if len(self.tickers) == 0:
            logger.info("All tickers already processed!")
            self.save_results_to_csv(0)
            return

        total = len(self.tickers)
        processed = 0
        start_time = time.time()

        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            future_to_ticker = {executor.submit(self.process_single, ticker): ticker for ticker in self.tickers}

            for future in as_completed(future_to_ticker):
                ticker = future_to_ticker[future]
                try:
                    result = future.result(timeout=60)
                    self.results.append(result)
                    processed += 1

                    # Progress every 10 tickers
                    if processed % 10 == 0:
                        elapsed = time.time() - start_time
                        eta = (elapsed / processed) * (total - processed) if processed > 0 else 0
                        logger.info(f"Progress: {processed}/{total} ({processed/total:.1%}). ETA: {eta/60:.1f} min.")

                    # Checkpoint every 50 tickers
                    if processed % self.checkpoint_interval == 0:
                        self.save_checkpoint()

                except Exception as e:
                    logger.error(f"Error processing {ticker}: {e}")
                    self.failed_tickers.append({'ticker': ticker, 'error': str(e)})

        elapsed_total = time.time() - start_time
        self.save_results_to_csv(elapsed_total)
        self.save_checkpoint()
        self.print_summary(elapsed_total)

    def save_results_to_csv(self, elapsed_time):
        """Flatten and save results to CSV with source attribution."""
        if not self.results:
            logger.warning("No results to save.")
            return

        flat_rows = []
        for r in self.results:
            flat_row = {'ticker': r['ticker']}
            for field in ['market_cap', 'trailing_pe', 'forward_pe', 'peg_ratio',
                          'short_pct', 'insider_pct', 'institution_pct', 'beta', 'price_to_book', 'price_to_sales']:
                data = r.get(field, {})
                flat_row[f"{field}_value"] = data.get('value')
                flat_row[f"{field}_source"] = data.get('source')
            flat_rows.append(flat_row)

        df = pd.DataFrame(flat_rows)
        
        # Add metadata
        metadata = (
            f"# Generated: {datetime.utcnow().isoformat()}\n"
            f"# Total tickers: {len(self.results) + len(self.failed_tickers)}\n"
            f"# Successfully processed: {len(self.results)}\n"
            f"# Failed: {len(self.failed_tickers)}\n"
            f"# Processing time: {elapsed_time:.1f} seconds ({elapsed_time/60:.1f} minutes)\n"
            f"# Success rate: {len(self.results)/(len(self.results)+len(self.failed_tickers)):.1%}\n"
        )

        with open(self.final_output_file, 'w') as f:
            f.write(metadata)
            df.to_csv(f, index=False)

        logger.info(f"Results saved to {self.final_output_file}")

    def print_summary(self, elapsed_time):
        """Print final summary report."""
        print("\n" + "="*60)
        print("BATCH PROCESSING SUMMARY")
        print("="*60)
        print(f"Total tickers attempted: {len(self.results) + len(self.failed_tickers)}")
        print(f"Successfully processed:  {len(self.results)}")
        print(f"Failed:                  {len(self.failed_tickers)}")
        print(f"Total time:              {elapsed_time/60:.1f} minutes")
        if len(self.results) + len(self.failed_tickers) > 0:
            success_rate = len(self.results)/(len(self.results) + len(self.failed_tickers))
            print(f"Success rate:            {success_rate:.1%}")
        print("="*60)

# ==================== RUN ON ALL 353 TICKERS ====================
# UNCOMMENT BELOW TO RUN FULL BATCH (will take 45-60 minutes)

print("⚠️ Full batch processing: 353 tickers, 45-60 minutes")
print("   Checkpoints saved every 50 tickers for recovery\n")

# Uncomment to run:
# processor = FundamentalDataBatchProcessor(
#     tickers=TICKERS,
#     max_workers=5,
#     checkpoint_interval=50
# )
# processor.run_batch()

# For now, test on 20 tickers to validate:
print("🔄 Testing batch processor on first 20 tickers...")
processor = FundamentalDataBatchProcessor(
    tickers=TICKERS[:20],
    max_workers=3,
    checkpoint_interval=50
)
processor.run_batch()
print("\n✅ Batch processor test complete!")

In [ ]:
def get_sec_insider_trades(ticker):
    """
    Get insider trading data from SEC EDGAR using their JSON API
    100% free, no API key needed
    """
    try:
        # Step 1: Get CIK (company ID) from ticker
        cik_lookup_url = "https://www.sec.gov/files/company_tickers.json"
        cik_data = requests.get(cik_lookup_url, headers=HEADERS, timeout=10).json()
        
        cik = None
        for entry in cik_data.values():
            if entry['ticker'].upper() == ticker.upper():
                cik = str(entry['cik_str']).zfill(10)  # Pad with zeros
                break
        
        if not cik:
            return {'ticker': ticker, 'insider_trades': 0, 'recent_filings': []}
        
        # Step 2: Get filing history from SEC
        filing_url = f"https://data.sec.gov/submissions/CIK{cik}.json"
        filing_data = requests.get(filing_url, headers=HEADERS, timeout=10).json()
        
        # Step 3: Filter for Form 4 (insider trades) and 8-K (major events)
        filings_df = pd.DataFrame(filing_data['filings']['recent'])
        insider_filings = filings_df[filings_df['form'].isin(['4', '8-K'])].head(10)
        
        recent_trades = []
        for _, filing in insider_filings.iterrows():
            recent_trades.append({
                'date': filing['filingDate'],
                'form': filing['form'],
                'description': filing.get('primaryDocDescription', 'N/A')
            })
        
        return {
            'ticker': ticker,
            'cik': cik,
            'insider_trades': len(insider_filings[insider_filings['form'] == '4']),
            'recent_filings': recent_trades
        }
        
    except Exception as e:
        return {'ticker': ticker, 'error': str(e), 'insider_trades': 0}

# Test on 3 tickers
print("🔄 Testing SEC insider scraper...")
test_insiders = []
for ticker in ['NVDA', 'IONQ', 'HOOD']:
    print(f"  Fetching {ticker} SEC filings...")
    insider = get_sec_insider_trades(ticker)
    test_insiders.append(insider)
    time.sleep(2)  # SEC rate limit

for ins in test_insiders:
    print(f"\n{ins['ticker']}: {ins.get('insider_trades', 0)} recent Form 4 filings")
    if 'recent_filings' in ins:
        for filing in ins['recent_filings'][:3]:
            print(f"  - {filing['date']} [{filing['form']}]")

## 📰 PHASE 6: News Scraping (Google News RSS)

In [ ]:
def scrape_google_news(ticker, max_articles=5):
    """
    Scrape Google News RSS feed for ticker
    Free, no API key needed
    """
    url = f"https://news.google.com/rss/search?q={ticker}+stock&hl=en-US&gl=US&ceid=US:en"
    
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(resp.content, features='xml')
        
        news_items = []
        for item in soup.findAll('item')[:max_articles]:
            news_items.append({
                'ticker': ticker,
                'title': item.title.text if item.title else '',
                'published': item.pubDate.text if item.pubDate else '',
                'link': item.link.text if item.link else ''
            })
        
        return news_items
        
    except Exception as e:
        return [{'ticker': ticker, 'error': str(e)}]

# Test on 3 tickers
print("🔄 Testing Google News scraper...")
all_news = []
for ticker in ['NVDA', 'IONQ', 'TSLA']:
    print(f"  Fetching news for {ticker}...")
    news = scrape_google_news(ticker, max_articles=3)
    all_news.extend(news)
    time.sleep(1)

news_df = pd.DataFrame(all_news)
print(f"\n✅ Collected {len(news_df)} news articles")
print("\nSample headlines:")
for _, row in news_df.head(5).iterrows():
    if 'title' in row:
        print(f"  {row['ticker']}: {row['title'][:70]}...")

## 🚀 PHASE 7: GPU-Accelerated Sentiment Analysis

In [ ]:
# Load FinBERT model for financial sentiment analysis
from transformers import pipeline

print("Loading FinBERT model for GPU sentiment analysis...")
device = 0 if torch.cuda.is_available() else -1
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    device=device,
    truncation=True,
    max_length=512
)

print(f"✅ Model loaded on {'GPU' if device == 0 else 'CPU'}")

# Test GPU speed
test_texts = [
    "Company reports record earnings with 50% revenue growth",
    "Stock plunges on disappointing guidance and regulatory concerns",
    "New product launch expected to drive significant market share gains"
]

start = time.time()
test_results = sentiment_pipeline(test_texts, batch_size=3)
elapsed = time.time() - start

print(f"\n⚡ GPU Speed Test: {len(test_texts)} texts in {elapsed:.3f}s ({len(test_texts)/elapsed:.1f} texts/sec)")
print("\nSample results:")
for text, result in zip(test_texts, test_results):
    print(f"  {result['label']}: {text[:50]}...")

In [ ]:
def analyze_sentiment_batch(news_list, batch_size=32):
    """
    GPU-accelerated batch sentiment analysis
    Takes list of news items and returns sentiment scores
    """
    if not news_list or len(news_list) == 0:
        return []
    
    # Extract headlines
    texts = [item.get('title', '') for item in news_list if item.get('title')]
    
    if not texts:
        return []
    
    # Batch process on GPU
    print(f"🔄 Analyzing {len(texts)} headlines on GPU...")
    start = time.time()
    results = sentiment_pipeline(texts, batch_size=batch_size)
    elapsed = time.time() - start
    
    print(f"✅ Processed {len(texts)} texts in {elapsed:.2f}s ({len(texts)/elapsed:.1f} texts/sec)")
    
    # Combine with original data
    for i, item in enumerate(news_list[:len(results)]):
        if 'title' in item:
            sentiment = results[i]
            item['sentiment_label'] = sentiment['label']
            item['sentiment_score'] = sentiment['score']
            # Convert to numeric: positive=1, negative=-1, neutral=0
            if sentiment['label'] == 'positive':
                item['sentiment_numeric'] = sentiment['score']
            elif sentiment['label'] == 'negative':
                item['sentiment_numeric'] = -sentiment['score']
            else:
                item['sentiment_numeric'] = 0
    
    return news_list

# Test GPU sentiment on news
if len(all_news) > 0:
    print("\n🚀 Running GPU sentiment analysis on news...")
    analyzed_news = analyze_sentiment_batch(all_news, batch_size=16)
    
    # Show results
    sentiment_df = pd.DataFrame(analyzed_news)
    print("\nSentiment Results:")
    print(sentiment_df[['ticker', 'sentiment_label', 'sentiment_score', 'title']].head(10))

## 🎯 PHASE 8: Master Collection Function (All Data Sources)

In [ ]:
def collect_all_data_for_ticker(ticker):
    """
    Master function: Collect ALL data for a single ticker
    Returns: dict with OHLCV, fundamentals, insider, sentiment
    """
    print(f"📊 {ticker}...", end=' ')
    
    data = {'ticker': ticker}
    
    try:
        # 1. OHLCV from yfinance
        ohlcv = yf.download(ticker, period='2y', progress=False)
        if not ohlcv.empty:
            data['bars_count'] = len(ohlcv)
            data['last_price'] = float(ohlcv['Close'].iloc[-1])
            data['avg_volume'] = float(ohlcv['Volume'].mean())
            data['volatility'] = float(ohlcv['Close'].pct_change().std() * np.sqrt(252))
            # Momentum
            data['return_1m'] = float((ohlcv['Close'].iloc[-1] / ohlcv['Close'].iloc[-20] - 1) * 100)
            data['return_3m'] = float((ohlcv['Close'].iloc[-1] / ohlcv['Close'].iloc[-60] - 1) * 100)
        
        time.sleep(0.3)
        
        # 2. Fundamentals from Yahoo scraping
        fundamentals = scrape_yahoo_fundamentals(ticker)
        data.update(fundamentals)
        time.sleep(1)
        
        # 3. Insider trades from SEC
        insider = get_sec_insider_trades(ticker)
        data['insider_trades_count'] = insider.get('insider_trades', 0)
        time.sleep(2)
        
        # 4. News sentiment
        news = scrape_google_news(ticker, max_articles=5)
        if news and len(news) > 0:
            analyzed = analyze_sentiment_batch(news, batch_size=5)
            if analyzed:
                sentiments = [n.get('sentiment_numeric', 0) for n in analyzed if 'sentiment_numeric' in n]
                data['news_count'] = len(analyzed)
                data['avg_sentiment'] = np.mean(sentiments) if sentiments else 0
                data['sentiment_std'] = np.std(sentiments) if len(sentiments) > 1 else 0
        time.sleep(0.5)
        
        print("✓")
        
    except Exception as e:
        print(f"✗ ({str(e)[:30]})")
        data['error'] = str(e)
    
    return data

# TEST on 10 tickers
print("\\n🔬 FULL DATA COLLECTION TEST (10 tickers)\\n")
test_batch = TICKERS[:10]
all_ticker_data = []

for ticker in test_batch:
    ticker_data = collect_all_data_for_ticker(ticker)
    all_ticker_data.append(ticker_data)

# Convert to dataframe
results_df = pd.DataFrame(all_ticker_data)
print(f"\\n✅ Collected data for {len(results_df)} tickers")
print("\\nSample results:")
print(results_df[['ticker', 'last_price', 'return_1m', 'avg_sentiment', 'insider_trades_count']].head())

## 📊 PHASE 9: Quantitative Screening & Scoring

In [ ]:
def calculate_composite_score(df):
    """
    Calculate multi-factor score for ranking stocks
    
    Factors:
    1. Momentum (30%): 1-month and 3-month returns
    2. Sentiment (20%): News sentiment score
    3. Insider Activity (15%): Recent Form 4 filings
    4. Volatility (15%): Higher vol = more opportunity
    5. Value (10%): P/E ratio (lower better)
    6. Short Interest (10%): High short % = squeeze potential
    """
    
    scores = pd.DataFrame(index=df.index)
    
    # 1. Momentum Score (30%)
    df['return_1m_clean'] = pd.to_numeric(df['return_1m'], errors='coerce')
    df['return_3m_clean'] = pd.to_numeric(df['return_3m'], errors='coerce')
    
    scores['momentum'] = (
        df['return_1m_clean'].fillna(0) * 0.6 + 
        df['return_3m_clean'].fillna(0) * 0.4
    )
    
    # 2. Sentiment Score (20%)
    scores['sentiment'] = df['avg_sentiment'].fillna(0) * 100
    
    # 3. Insider Activity Score (15%)
    scores['insider'] = df['insider_trades_count'].fillna(0) * 10
    
    # 4. Volatility Score (15%) - normalized
    df['volatility_clean'] = pd.to_numeric(df['volatility'], errors='coerce')
    scores['volatility'] = df['volatility_clean'].fillna(0) * 100
    
    # 5. Value Score (10%) - inverse P/E
    df['pe_clean'] = df['trailing_pe'].replace('N/A', np.nan)
    df['pe_clean'] = pd.to_numeric(df['pe_clean'], errors='coerce')
    scores['value'] = 100 / (df['pe_clean'].fillna(50) + 1)  # Lower P/E = higher score
    
    # 6. Short Interest Score (10%)
    df['short_clean'] = df['short_pct'].replace('N/A', np.nan)
    df['short_clean'] = pd.to_numeric(df['short_clean'].str.rstrip('%'), errors='coerce')
    scores['short_squeeze'] = df['short_clean'].fillna(0) * 2
    
    # Normalize all scores to 0-100
    for col in scores.columns:
        min_val = scores[col].min()
        max_val = scores[col].max()
        if max_val > min_val:
            scores[col] = (scores[col] - min_val) / (max_val - min_val) * 100
    
    # Composite score with weights
    weights = {
        'momentum': 0.30,
        'sentiment': 0.20,
        'insider': 0.15,
        'volatility': 0.15,
        'value': 0.10,
        'short_squeeze': 0.10
    }
    
    df['composite_score'] = sum(scores[factor] * weight for factor, weight in weights.items())
    
    # Add individual factor scores
    for factor in scores.columns:
        df[f'{factor}_score'] = scores[factor]
    
    return df

# Apply scoring to test data
if len(results_df) > 0:
    print("🎯 Calculating composite scores...")
    scored_df = calculate_composite_score(results_df.copy())
    
    # Rank by score
    scored_df = scored_df.sort_values('composite_score', ascending=False)
    
    print("\n🏆 TOP RANKED TICKERS:")
    print(scored_df[['ticker', 'composite_score', 'momentum_score', 
                     'sentiment_score', 'insider_score', 'last_price']].head(10))

## 🚀 PHASE 10: FULL PIPELINE - Process All 353 Tickers

In [ ]:
# WARNING: This will take 2-3 hours for 353 tickers
# Uncomment to run full pipeline

# print("=" * 60)
# print("🚀 RUNNING FULL PIPELINE ON 353 TICKERS")
# print("=" * 60)
# print(f"Started: {datetime.now()}")
# print(f"Estimated time: 2-3 hours\\n")

# all_results = []
# failed_tickers = []

# for i, ticker in enumerate(TICKERS, 1):
#     print(f"[{i}/{len(TICKERS)}] ", end='')
#     try:
#         data = collect_all_data_for_ticker(ticker)
#         all_results.append(data)
#     except Exception as e:
#         print(f"{ticker} FAILED: {e}")
#         failed_tickers.append(ticker)
    
#     # Save checkpoint every 50 tickers
#     if i % 50 == 0:
#         checkpoint_df = pd.DataFrame(all_results)
#         checkpoint_df.to_csv(f'{DATA_DIR}/checkpoint_{i}.csv', index=False)
#         print(f"\\n📁 Checkpoint saved: {i} tickers\\n")

# # Final save
# final_df = pd.DataFrame(all_results)
# final_df.to_csv(f'{DATA_DIR}/all_353_tickers_raw.csv', index=False)

# print(f"\\n✅ COMPLETE at {datetime.now()}")
# print(f"Success: {len(all_results)}, Failed: {len(failed_tickers)}")

print("⚠️ FULL PIPELINE READY - Uncomment above code to run")
print("💡 Recommended: Start with 30-50 tickers first to validate")

## 🏆 PHASE 11: Extract Top 50 Tickers

In [ ]:
# After running full pipeline, use this to extract top 50

# # Load raw data
# full_df = pd.read_csv(f'{DATA_DIR}/all_353_tickers_raw.csv')

# # Calculate scores
# scored_df = calculate_composite_score(full_df)

# # Rank and select top 50
# top50 = scored_df.nlargest(50, 'composite_score')

# # Save results
# top50.to_csv(OUTPUT_FILE, index=False)

# print("🏆 TOP 50 TICKERS SELECTED")
# print("\nBreakdown by sector:")
# print(top50['sector'].value_counts())
# print("\nTop 10:")
# print(top50[['ticker', 'composite_score', 'sector', 'last_price', 
#              'return_1m', 'avg_sentiment']].head(10))

# # Create watchlist file
# with open(f'{DATA_DIR}/top50_watchlist.txt', 'w') as f:
#     for ticker in top50['ticker']:
#         f.write(f"{ticker}\\n")

# print(f"\n✅ Saved to: {OUTPUT_FILE}")
# print(f"✅ Watchlist: {DATA_DIR}/top50_watchlist.txt")

print("⚠️ Run after full pipeline completes")